# 07 APD dataset: both cell lines in the same videos

HeLa and NIH3T3 cells were mixed and recorded together in flow with the Phantom TMX in Ph2. One cell line was
stained (NIH3T3 on 161026, HeLa on 201026), and every track was labelled from its association with the APD signal:
tracks with an APD peak belong to the stained line, tracks without a peak to the unstained line. Tracks without a
clear assignment are not used. The labels were not validated independently.

1. **Grouped cross-validation over videos** (five folds): all cells of a video are in the same part. Validation
   (epoch selection) uses single cells of the training videos.
2. **Accuracy of stained and unstained cells** per cell line, in the cross-validation.
3. **Held-out acquisition day:** trained on one day, tested on the other. Because the stained line differs between
   the days, this split also reverses the staining.
4. **Transfer from the static Phantom Ph2 images:** trained on the static Phantom images acquired in Ph2, tested on
   all APD cells. Here the size filter is applied to the Ph2 images only.

Models: chip network (8 × 8) and CNN (64 × 64), three seeds each, and the feature control.

Thesis: Results, Section "Classification of the APD dataset" (Table `tab:res-apd-classification`).

Output: `results/07_apd_cv_*`, `07_apd_day_*`, `07_static_ph2_to_apd_*`.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import display

import thesis_config as C     # paths and settings
import thesis_ml as M         # data, preprocessing, models, training and evaluation

pd.set_option('display.precision', 3)
C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results folder:', C.RESULTS_DIR.resolve())
if C.FAST:
    print('QUICK TEST (THESIS_FAST=1): few images and 3 epochs, the numbers are not meaningful.')

## 1. Data

In [ ]:
data = C.load_data(sets=['flow_apd', 'phantom_static'], ps_modes=['Ph2'])
A, PS = data['flow_apd'], data['phantom_static']
print('Stained cell line per day:', C.STAINED)
display(A.meta.groupby(['date', 'video', 'label']).size().unstack('label', fill_value=0))
RN = {'recall_0': f'recall {C.CLASSES[0]}', 'recall_1': f'recall {C.CLASSES[1]}'}

## 2. Grouped cross-validation over videos

In [ ]:
folds = M.grouped_folds(A.meta.y.to_numpy(), A.meta.group_video.to_numpy(), C.N_FOLDS, seed=0)
display(M.split_table(A.meta, 'group_video', folds).pivot_table(index='fold', columns='part',
        values=['n_groups', 'n_HeLa', 'n_NIH3T3']))
cv_m, cv_p = [], []
for kind, size in C.MODELS:
    m, p, _ = M.run_grouped_cv(A.meta, A.X, kind, size, 'group_video', C.TRAIN, None, C.SEEDS_BY_MODEL[kind],
                               C.VAL_FRAC, scenario='apd_cv', val_group_col='group_cell', folds=folds)
    cv_m.append(m); cv_p.append(p)
cv_m.append(M.run_control_cv(A.meta, A.feat, 'group_video', None, scenario='apd_cv', folds=folds))
cv_metrics, cv_preds = pd.concat(cv_m, ignore_index=True), pd.concat(cv_p, ignore_index=True)
M.save_results('07_apd_cv', C.RESULTS_DIR, cv_metrics, cv_preds)
display(cv_metrics.groupby('model').macro_f1.agg(['mean', 'std', 'count']).round(3))
display(cv_metrics.groupby('model')[['recall_0', 'recall_1']].mean().rename(columns=RN).round(3))

### Stained and unstained cells
Fraction of correctly classified cells per cell line, split by whether the cell was assigned to the stained group
(APD peak). Predictions of all folds and seeds.

In [ ]:
def staining_table(preds, by=('model', 'label')):
    p = preds.merge(A.meta[['uid', 'stained']], on='uid', how='left')
    p['stained'] = p.stained.map({1.0: 'stained', 0.0: 'not stained'})
    return p.groupby([*by, 'stained']).agg(n=('uid', 'size'), accuracy=('correct', 'mean')).unstack('stained')


display(staining_table(cv_preds).round(3))

## 3. Held-out acquisition day
One fold per day: trained on the other day, tested on this day. Validation uses single cells of the training day.

In [ ]:
days = sorted(A.meta.date.unique())
folds = [(np.flatnonzero(A.meta.date.ne(d)), np.flatnonzero(A.meta.date.eq(d))) for d in days]
names = [f'test {d}' for d in days]
day_m, day_p = [], []
for kind, size in C.MODELS:
    m, p, _ = M.run_grouped_cv(A.meta, A.X, kind, size, 'date', C.TRAIN, None, C.SEEDS_BY_MODEL[kind], C.VAL_FRAC,
                               scenario='apd_day', val_group_col='group_cell', folds=folds, fold_names=names)
    day_m.append(m); day_p.append(p)
day_m.append(M.run_control_cv(A.meta, A.feat, 'date', None, scenario='apd_day', folds=folds, fold_names=names))
day_metrics, day_preds = pd.concat(day_m, ignore_index=True), pd.concat(day_p, ignore_index=True)
M.save_results('07_apd_day', C.RESULTS_DIR, day_metrics, day_preds)
display(day_metrics.groupby(['model', 'fold_name'])[['macro_f1', 'roc_auc', 'recall_0', 'recall_1']].mean()
        .rename(columns=RN).round(3))
display(staining_table(day_preds, by=('model', 'date', 'label')).round(3))

## 4. Transfer from the static Phantom Ph2 images
Trained on the static Phantom images acquired in Ph2 (epoch selection on validation groups of the static images),
tested on all APD cells. Macro-F1: mean and range over the seeds.

In [ ]:
tr_m, tr_p = [], []
for kind, size in C.MODELS:
    m, p = M.run_transfer(PS.meta, PS.X, {'flow_apd': (A.meta, A.X)}, kind, size, C.PS_GROUP_COL, C.TRAIN,
                          C.SEEDS_BY_MODEL[kind], C.VAL_FRAC, scenario='static_ph2_to_apd')
    tr_m.append(m); tr_p.append(p)
tr_m.append(M.control_transfer(PS.meta, PS.feat, {'flow_apd': (A.meta, A.feat)}, C.PS_GROUP_COL,
                               scenario='static_ph2_to_apd'))
tr_metrics, tr_preds = pd.concat(tr_m, ignore_index=True), pd.concat(tr_p, ignore_index=True)
M.save_results('07_static_ph2_to_apd', C.RESULTS_DIR, tr_metrics, tr_preds)
display(PS.meta.groupby('label').size().rename('static Ph2 training images'))
display(tr_metrics.groupby('model').macro_f1.agg(['mean', 'min', 'max', 'count']).round(3))